# آزمایش‌های مقاله: معماری × ρ × Dropout روی Colored MNIST (با آمار دقیق)

این نوت‌بوک آزمایش‌های لازم برای تبدیل پروژه درسی به یک مطالعه علمی را اجرا می‌کند:
- **جاروب معماری × ρ:** MLP، CNN با ۱، ۲ و ۳ لایه کانولوشن؛ ρ ∈ {0.5, 0.7, 0.9, 0.95, 0.99, 1.0}.
- **جاروب Dropout:** {0, 0.2, 0.5, 0.7} برای MLP و CNN۳ در ρ ∈ {0.9, 0.99}.
- **«حذف ۰/۵/۱۰/۲۰/۵۰٪ نمونه مخالف‌جهت»** همان جاروب ρ است (ρ = ۱ − درصد مخالف‌جهت) و جدا اجرا نمی‌شود؛ ρ=1.0 همان «۰٪ مخالف‌جهت» است.
- **۵ seed** برای هر پیکربندی. رنگ‌آمیزی داده هم با seed عوض می‌شود تا تغییرپذیری داده در بازه اطمینان ۹۵٪ (توزیع t) لحاظ شود.
- **آمار:** آزمون Welch بین معماری‌ها با تصحیح Holm و اندازه اثر Cohen's d. (paired t-test استفاده نشده، چون seedها جفت واقعی نیستند.)
- **تحلیل تئوریک:** بررسی تجربی Acc(ρ′) = ρ′·a + (1−ρ′)·c و اتحاد ID−OOD2 = (ρ−0.1)(a−c).
- **تفسیرپذیری:** Integrated Gradients (پایه سیاه و پایه Counterfactual با رنگ تعویض‌شده + کنترل Completeness)، اثر تعویض رنگ روی لاجیت، ماتریس درهم‌ریختگی OOD-2 و خطای هر رقم.
- **خروجی‌ها:** جدول‌های CSV و `.tex`، شکل‌ها با ۳۰۰ dpi و `raw_results.json` در `paper_results/`.

**نکات مهم:**
1. **فرضیه‌ها را قبل از اجرا ثابت کنید** (H1: MLP افت بیشتری از CNN دارد؛ H2: افت با ρ زیاد می‌شود؛ H3: Dropout کمتر ⇒ افت بیشتر؛ H4: Reweighting بهترین روش است) و هر کدام را «تأیید» یا «رد» گزارش کنید. این‌ها فرضیه‌اند، نه نتیجه؛ ممکن است رد شوند. در پروژه شما CNN سه‌لایه افت ID−OOD2 فقط ۱.۴ واحد داشت، پس رد شدن برخی فرضیه‌ها محتمل است و یافته معتبری است.
2. شبکه آزمایش‌ها قبل از اجرا ثابت است؛ اگر بعد از دیدن نتایج پیکربندی اضافه می‌کنید، در مقاله بنویسید که اکتشافی (post-hoc) بوده است.
3. اجرا **قابل ازسرگیری** است: اگر Colab قطع شد، همان سلول اجرا را دوباره اجرا کنید؛ پیکربندی‌های تمام‌شده از `raw_results.json` خوانده می‌شوند. برای محافظت در برابر پاک‌شدن دیسک، پوشه `paper_results` را روی Google Drive بگذارید.
4. زمان تقریبی: ۳۶ پیکربندی × ۵ seed = ۱۸۰ اجرا، حدود ۲ تا ۳ ساعت روی T4. با `QUICK_TEST = True` در چند دقیقه اجرا می‌شود.
5. این کد اجرا نشده است (فقط نحو و سلول‌های تحلیل روی داده مصنوعی تست شده‌اند)؛ اگر خطا دیدید متنش را بفرستید.

In [ ]:
import os, json, time, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# ===================== تنظیمات طراحی آزمایش (قبل از اجرا ثابت کنید و تغییر ندهید) =====================
QUICK_TEST  = False                                   # True → تست سریع (۲ epoch، ۲ seed، شبکه کوچک آزمایش‌ها)
SEEDS       = [0, 1] if QUICK_TEST else [0, 1, 2, 3, 4]   # حداقل ۵ seed برای بازه اطمینان
EPOCHS      = 2 if QUICK_TEST else 20
BATCH, LR   = 64, 1e-3
DEF_DROP    = 0.5                                     # Dropout پیش‌فرض لایه fc1 برای همه معماری‌ها
ARCHS       = ['MLP', 'CNN1', 'CNN2', 'CNN3']
RHOS        = [0.5, 0.9, 0.99] if QUICK_TEST else [0.5, 0.7, 0.9, 0.95, 0.99, 1.0]
DROPS       = [0.0, 0.5] if QUICK_TEST else [0.0, 0.2, 0.5, 0.7]
DROP_ARCHS  = ['MLP', 'CNN3']
DROP_RHOS   = [0.9] if QUICK_TEST else [0.9, 0.99]
OUT         = "paper_results"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def all_configs():
    """فهرست (معماری، ρ، dropout). آزمایش A: معماری×ρ ؛ آزمایش B: dropout. (حذف ۰/۵/۱۰/۲۰/۵۰٪ نمونه مخالف‌جهت همان جاروب ρ است: ρ = 1 − درصد مخالف‌جهت)"""
    cfgs = [(a, r, DEF_DROP) for a in ARCHS for r in RHOS]
    cfgs += [(a, r, dp) for a in DROP_ARCHS for r in DROP_RHOS for dp in DROPS]
    return list(dict.fromkeys(cfgs))
CFGS = all_configs()
print(len(CFGS), "configs ×", len(SEEDS), "seeds =", len(CFGS) * len(SEEDS), "runs")

## داده

In [ ]:
# ---------------- داده: Colored MNIST (دورنگ) با ρ متغیر ----------------
mn_tr = datasets.MNIST('./data', train=True,  download=True)
mn_te = datasets.MNIST('./data', train=False, download=True)
G_tr = mn_tr.data.numpy().astype(np.float32) / 255.0; Y_tr = mn_tr.targets.numpy().astype(np.int64)
G_te = mn_te.data.numpy().astype(np.float32) / 255.0; Y_te = mn_te.targets.numpy().astype(np.int64)

def make_colors(y, rho, seed):
    """0=قرمز، 1=سبز؛ ارقام 0-4 با احتمال rho قرمز و 5-9 با احتمال rho سبز."""
    rng = np.random.RandomState(seed)
    base = (y >= 5).astype(np.int64)
    return np.where(rng.rand(len(y)) < rho, base, 1 - base)

class Split:
    """تصویر خاکستری + رنگ روی GPU؛ تصویر RGB فقط به‌صورت دسته‌ای ساخته می‌شود (کم‌مصرف)."""
    def __init__(self, gray, y, c):
        self.gray = torch.from_numpy(gray).to(DEVICE)
        self.y = torch.from_numpy(y).long().to(DEVICE); self.c = torch.from_numpy(c).long().to(DEVICE)
        self.aligned = (self.c == (self.y >= 5).long())
    def __len__(self): return len(self.y)
    def rgb(self, idx=None):
        g, c = (self.gray, self.c) if idx is None else (self.gray[idx], self.c[idx])
        out = torch.zeros(len(g), 3, 28, 28, device=DEVICE)
        out[torch.arange(len(g), device=DEVICE), c] = g
        return out

_perm = np.random.RandomState(42).permutation(len(G_tr))
_TR, _VA = _perm[:50000], _perm[50000:]            # تقسیم ثابت ارقام آموزش/اعتبارسنجی

def build_data(rho, seed):
    """رنگ‌آمیزی به seed وابسته است تا تغییرپذیری ناشی از داده هم در بازه اطمینان لحاظ شود.
    سه مجموعه تست از همان ۱۰٬۰۰۰ رقم test ساخته می‌شوند (ID: ρ، OOD-1: 0.5، OOD-2: 0.1)."""
    return {'train': Split(G_tr[_TR], Y_tr[_TR], make_colors(Y_tr[_TR], rho, 1000 + seed)),
            'val':   Split(G_tr[_VA], Y_tr[_VA], make_colors(Y_tr[_VA], rho, 1100 + seed)),
            'id':    Split(G_te, Y_te, make_colors(Y_te, rho, 1200 + seed)),
            'ood1':  Split(G_te, Y_te, make_colors(Y_te, 0.5, 1300 + seed)),
            'ood2':  Split(G_te, Y_te, make_colors(Y_te, 0.1, 1400 + seed))}

## معماری‌ها و آموزش

In [ ]:
# ---------------- معماری‌ها: MLP، CNN۱ (یک کانولوشن)، CNN۲ (دو کانولوشن)، CNN۳ (سه کانولوشن) ----------------
class MLP(nn.Module):
    def __init__(self, dp):
        super().__init__()
        self.fc1, self.fc2, self.drop = nn.Linear(3 * 28 * 28, 256), nn.Linear(256, 10), nn.Dropout(dp)
    def forward(self, x):
        return self.fc2(self.drop(F.relu(self.fc1(x.flatten(1)))))

class CNN1(nn.Module):
    def __init__(self, dp):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.pool = nn.MaxPool2d(2)
        self.fc1, self.fc2, self.drop = nn.Linear(32 * 14 * 14, 256), nn.Linear(256, 10), nn.Dropout(dp)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        return self.fc2(self.drop(F.relu(self.fc1(x.flatten(1)))))

class CNN2(nn.Module):
    def __init__(self, dp):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.pool = nn.MaxPool2d(2)
        self.fc1, self.fc2, self.drop = nn.Linear(64 * 7 * 7, 256), nn.Linear(256, 10), nn.Dropout(dp)
    def forward(self, x):
        x = self.pool(F.relu(self.conv2(self.pool(F.relu(self.conv1(x))))))   # 28→14→7
        return self.fc2(self.drop(F.relu(self.fc1(x.flatten(1)))))

class CNN3(nn.Module):
    def __init__(self, dp):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1); self.pool = nn.MaxPool2d(2)
        self.fc1, self.fc2, self.drop = nn.Linear(128 * 3 * 3, 256), nn.Linear(256, 10), nn.Dropout(dp)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))); x = self.pool(F.relu(self.conv2(x))); x = F.relu(self.conv3(x))
        return self.fc2(self.drop(F.relu(self.fc1(self.pool(x).flatten(1)))))

def make_model(arch, dp):
    return {'MLP': MLP, 'CNN1': CNN1, 'CNN2': CNN2, 'CNN3': CNN3}[arch](dp).to(DEVICE)

for a in ARCHS:
    print(a, f"{sum(p.numel() for p in make_model(a, DEF_DROP).parameters()):,} parameters")
    assert make_model(a, DEF_DROP)(torch.randn(2, 3, 28, 28, device=DEVICE)).shape == (2, 10), f"shape check failed: {a}"   # بررسی سریع شکل‌ها قبل از شروع جاروب

@torch.no_grad()
def logits_split(model, d, swap=False, bs=5000):
    model.eval(); out = []
    for i in range(0, len(d), bs):
        x = d.rgb(torch.arange(i, min(i + bs, len(d)), device=DEVICE))
        out.append(model(x[:, [1, 0, 2]] if swap else x))
    return torch.cat(out)

def evaluate(model, data):
    r = {k: (logits_split(model, data[k]).argmax(1) == data[k].y).float().mean().item() for k in ['train', 'val', 'id', 'ood1', 'ood2']}
    d = data['id']; lg = logits_split(model, d); ok = lg.argmax(1) == d.y
    r['id_aligned']  = ok[d.aligned].float().mean().item()
    r['id_conflict'] = ok[~d.aligned].float().mean().item() if (~d.aligned).any() else float('nan')
    lgs = logits_split(model, d, swap=True)                                   # همان تصویر با رنگ تعویض‌شده
    r['swap_acc'] = (lgs.argmax(1) == d.y).float().mean().item()
    r['color_effect'] = (lg.gather(1, d.y[:, None]) - lgs.gather(1, d.y[:, None])).mean().item()   # میانگین Δ لاجیت کلاس درست در اثر رنگ
    d2 = data['ood2']; p2 = logits_split(model, d2).argmax(1)
    r['conf_ood2'] = torch.bincount(d2.y * 10 + p2, minlength=100).reshape(10, 10).tolist()          # ماتریس درهم‌ریختگی OOD-2
    return r

def train_run(arch, dp, data, seed):
    set_seed(seed)
    model = make_model(arch, dp); opt = torch.optim.Adam(model.parameters(), lr=LR)
    tr = data['train']; N = len(tr)
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(N, device=DEVICE)
        for i in range(0, N, BATCH):
            idx = perm[i:i+BATCH]
            loss = F.cross_entropy(model(tr.rgb(idx)), tr.y[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return model

## اجرای جاروب (قابل ازسرگیری)

In [ ]:
# ---------------- اجرای جاروب (قابل ازسرگیری: اگر Colab قطع شد، همین سلول را دوباره اجرا کنید) ----------------
RES_PATH = f"{OUT}/raw_results.json"
RESULTS = json.load(open(RES_PATH, encoding='utf-8')) if os.path.exists(RES_PATH) else {}
KEEP = {}      # مدل‌های ρ=0.99 و seed اول برای تفسیرپذیری
def rkey(a, r, dp, s): return f"{a}|{r}|{dp}|{s}"

t_all = time.time()
for rho in sorted({c[1] for c in CFGS}):
    for seed in SEEDS:
        todo = [(a, dp) for (a, r, dp) in CFGS if r == rho and rkey(a, rho, dp, seed) not in RESULTS]
        if not todo: continue
        data = build_data(rho, seed)
        for a, dp in todo:
            t0 = time.time()
            model = train_run(a, dp, data, seed)
            RESULTS[rkey(a, rho, dp, seed)] = evaluate(model, data)
            json.dump(RESULTS, open(RES_PATH, "w", encoding="utf-8"))
            if rho == 0.99 and seed == SEEDS[0] and dp == DEF_DROP: KEEP[a] = model
            r_ = RESULTS[rkey(a, rho, dp, seed)]
            print(f"{a:5s} rho={rho:<5} drop={dp:<4} seed={seed}  ID={r_['id']:.4f} OOD1={r_['ood1']:.4f} OOD2={r_['ood2']:.4f}  ({time.time()-t0:.0f}s)")
        del data
print(f"total {time.time()-t_all:.0f}s, runs stored: {len(RESULTS)}")

## جدول‌ها (میانگین ± بازه اطمینان ۹۵٪)

In [ ]:
# ---------------- جدول‌ها: میانگین ± بازه اطمینان ۹۵٪ (توزیع t) ----------------
rows = []
for k, m in RESULTS.items():
    a, r, dp, s = k.split('|')
    rows.append(dict(arch=a, rho=float(r), dp=float(dp), seed=int(s), **{x: m[x] for x in ['train', 'val', 'id', 'ood1', 'ood2', 'id_aligned', 'id_conflict', 'swap_acc', 'color_effect']}))
df = pd.DataFrame(rows)
df['drop1'] = df['id'] - df['ood1']; df['drop2'] = df['id'] - df['ood2']
for c in ['train', 'val', 'id', 'ood1', 'ood2', 'id_aligned', 'id_conflict', 'swap_acc', 'drop1', 'drop2']: df[c] = df[c] * 100

def mean_ci(v):
    v = np.asarray(v, float); v = v[~np.isnan(v)]; n = len(v)
    if n == 0: return np.nan, np.nan
    if n < 2: return v.mean(), 0.0
    return v.mean(), stats.t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)
def fmt(v, d=2): m, h = mean_ci(v); return f"{m:.{d}f} ± {h:.{d}f}"

def to_latex(tab, caption, label):
    L = ["\\begin{table}[t]", "\\centering", "\\small", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
         "\\begin{tabular}{" + "l" * len(tab.columns) + "}", "\\toprule", " & ".join(str(c).replace('_', '\\_') for c in tab.columns) + " \\\\", "\\midrule"]
    for _, r in tab.iterrows(): L.append(" & ".join(str(x).replace('±', '$\\pm$').replace('_', '\\_') for x in r) + " \\\\")
    return "\n".join(L + ["\\bottomrule", "\\end{tabular}", "\\end{table}"])

main = df[df.dp == DEF_DROP]
tab_main = pd.DataFrame([dict(arch=a, rho=r, n_seeds=len(g), ID=fmt(g['id']), OOD1=fmt(g['ood1']), OOD2=fmt(g['ood2']), drop_ID_OOD2=fmt(g['drop2']))
                         for (a, r), g in main.groupby(['arch', 'rho'])])
tab_main['arch'] = pd.Categorical(tab_main['arch'], ARCHS); tab_main = tab_main.sort_values(['arch', 'rho']).reset_index(drop=True)
tab_main.to_csv(f"{OUT}/table_main_arch_rho.csv", index=False, encoding='utf-8-sig'); display(tab_main)
open(f"{OUT}/table_main_arch_rho.tex", "w").write(to_latex(tab_main, "Accuracy (\\%) on ID and OOD test sets (mean $\\pm$ 95\\% CI over seeds).", "tab:main"))

tab_dp = pd.DataFrame([dict(arch=a, rho=r, dropout=dp, ID=fmt(g['id']), OOD2=fmt(g['ood2']), drop_ID_OOD2=fmt(g['drop2']))
                       for (a, r, dp), g in df[df.arch.isin(DROP_ARCHS) & df.rho.isin(DROP_RHOS)].groupby(['arch', 'rho', 'dp'])])
tab_dp.to_csv(f"{OUT}/table_dropout.csv", index=False, encoding='utf-8-sig'); display(tab_dp)
open(f"{OUT}/table_dropout.tex", "w").write(to_latex(tab_dp, "Effect of dropout (mean $\\pm$ 95\\% CI).", "tab:dropout"))

## شکل‌ها

In [ ]:
# ---------------- شکل‌ها (۳۰۰ dpi، با نوار خطا) ----------------
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
for j, (col, nm) in enumerate([('id', 'ID'), ('ood1', 'OOD-1 (rho=0.5)'), ('ood2', 'OOD-2 (rho=0.1)')]):
    for a in ARCHS:
        g = main[main.arch == a].groupby('rho')[col]
        m = np.array([mean_ci(v)[0] for _, v in g]); h = np.array([mean_ci(v)[1] for _, v in g]); x = np.array([r for r, _ in g])
        ax[j].plot(x, m, marker='o', label=a); ax[j].fill_between(x, m - h, m + h, alpha=0.2)
    ax[j].set_xlabel('training rho'); ax[j].set_ylabel('accuracy (%)'); ax[j].set_title(nm)
ax[0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/fig_acc_vs_rho.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 3.8))
for a in ARCHS:
    g = main[main.arch == a].groupby('rho')['drop2']
    m = np.array([mean_ci(v)[0] for _, v in g]); h = np.array([mean_ci(v)[1] for _, v in g]); x = np.array([r for r, _ in g])
    ax.errorbar(x, m, yerr=h, marker='o', capsize=3, label=a)
ax.set_xlabel('training rho'); ax.set_ylabel('ID − OOD-2 accuracy (points)'); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig_drop_vs_rho.png", dpi=300); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for j, col in enumerate(['ood2', 'drop2']):
    for a in DROP_ARCHS:
        for r in DROP_RHOS:
            g = df[(df.arch == a) & (df.rho == r)].groupby('dp')[col]
            x = [d_ for d_, _ in g]; m = [mean_ci(v)[0] for _, v in g]; h = [mean_ci(v)[1] for _, v in g]
            ax[j].errorbar(x, m, yerr=h, marker='o', capsize=3, label=f"{a}, rho={r}")
    ax[j].set_xlabel('dropout'); ax[j].set_ylabel('OOD-2 accuracy (%)' if col == 'ood2' else 'ID − OOD-2 (points)')
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/fig_dropout.png", dpi=300); plt.show()

# اثر رنگ روی لاجیت کلاس درست و دقت پس از تعویض رنگ
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for j, col in enumerate(['color_effect', 'swap_acc']):
    for k_, a in enumerate(ARCHS):
        g = main[main.arch == a].groupby('rho')[col]; x = np.array([r for r, _ in g]); m = np.array([mean_ci(v)[0] for _, v in g]); h = np.array([mean_ci(v)[1] for _, v in g])
        ax[j].errorbar(x, m, yerr=h, marker='o', capsize=3, label=a)
    ax[j].set_xlabel('training rho'); ax[j].set_ylabel('mean logit drop after color swap' if col == 'color_effect' else 'accuracy after color swap (%)')
ax[0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/fig_color_effect.png", dpi=300); plt.show()

## آزمون‌های آماری (Welch + Holm)

In [ ]:
# ---------------- آزمون‌های آماری ----------------
# توجه: seedها جفت واقعی نیستند (رنگ‌آمیزی و مقداردهی اولیه با seed عوض می‌شود)، پس به‌جای paired t-test از Welch t-test استفاده می‌شود.
def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

rows = []
for r in sorted(main.rho.unique()):
    for i, a in enumerate(ARCHS):
        for b in ARCHS[i + 1:]:
            x = main[(main.rho == r) & (main.arch == a)]['ood2'].values; y = main[(main.rho == r) & (main.arch == b)]['ood2'].values
            if len(x) < 2 or len(y) < 2: continue
            t, p = stats.ttest_ind(x, y, equal_var=False)
            sp = math.sqrt((x.var(ddof=1) + y.var(ddof=1)) / 2); d_ = (x.mean() - y.mean()) / sp if sp > 0 else float('nan')
            rows.append(dict(rho=r, A=a, B=b, meanA=x.mean(), meanB=y.mean(), diff=x.mean() - y.mean(), p_welch=p, cohen_d=d_))
T_stat = pd.DataFrame(rows)
if len(T_stat):
    T_stat['p_holm'] = holm(T_stat['p_welch'].values)          # تصحیح Holm برای مقایسه‌های چندگانه
    T_stat['significant_0.05'] = T_stat['p_holm'] < 0.05
    T_stat.round(4).to_csv(f"{OUT}/table_stats_ood2.csv", index=False, encoding='utf-8-sig'); display(T_stat.round(4))
else:
    print("برای آزمون آماری حداقل ۲ seed لازم است.")

## تحلیل تئوریک: بررسی فرمول

In [ ]:
# ---------------- تحلیل تئوریک: Acc(ρ') ≈ ρ'·a + (1−ρ')·c  و  ID − OOD2 ≈ (ρ − 0.1)(a − c) ----------------
# a = دقت روی نمونه‌های هم‌جهت، c = دقت روی نمونه‌های مخالف‌جهت (روی تست ID). فرض: دقت گروهی بین مجموعه‌های تست یکسان است.
t = main.dropna(subset=['id_aligned', 'id_conflict']).copy()
t['pred_ood1'] = 0.5 * t.id_aligned + 0.5 * t.id_conflict
t['pred_ood2'] = 0.1 * t.id_aligned + 0.9 * t.id_conflict
t['pred_drop2'] = (t.rho - 0.1) * (t.id_aligned - t.id_conflict)
t['err_ood1'] = t.pred_ood1 - t.ood1; t['err_ood2'] = t.pred_ood2 - t.ood2; t['err_drop2'] = t.pred_drop2 - t.drop2
print("خطای پیش‌بینی فرمول (واحد درصد): میانگین مطلق / بیشینه مطلق")
for c in ['err_ood1', 'err_ood2', 'err_drop2']: print(f"  {c}: {t[c].abs().mean():.3f} / {t[c].abs().max():.3f}")
tab_theory = pd.DataFrame([{'arch': a_, 'rho': r_, 'a-c (shortcut gain)': fmt(g.id_aligned - g.id_conflict), 'OOD2 observed': fmt(g.ood2), 'OOD2 predicted': fmt(g.pred_ood2),
                            'drop observed': fmt(g.drop2), 'drop predicted': fmt(g.pred_drop2)} for (a_, r_), g in t.groupby(['arch', 'rho'])])
tab_theory.to_csv(f"{OUT}/table_theory_check.csv", index=False, encoding='utf-8-sig'); display(tab_theory)
fig, ax = plt.subplots(figsize=(4.8, 4.5))
for a in ARCHS:
    g = t[t.arch == a]; ax.scatter(g.ood2, g.pred_ood2, s=14, label=a)
lo, hi = min(t.ood2.min(), t.pred_ood2.min()) - 1, 100.5; ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('observed OOD-2 accuracy (%)'); ax.set_ylabel('predicted by rho·a+(1−rho)·c (%)'); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig_theory_check.png", dpi=300); plt.show()

## تجزیه خطا: ماتریس درهم‌ریختگی

In [ ]:
# ---------------- تجزیه خطا: ماتریس درهم‌ریختگی و خطای هر رقم روی OOD-2 ----------------
CONF_RHO = 0.99 if 0.99 in RHOS else max(RHOS)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6)); err_rows = []
for j, a in enumerate(['MLP', 'CNN3']):
    M = np.zeros((10, 10))
    for s in SEEDS:
        k = rkey(a, CONF_RHO, DEF_DROP, s)
        if k in RESULTS: M += np.array(RESULTS[k]['conf_ood2'])
    Mn = M / np.maximum(M.sum(1, keepdims=True), 1)
    im = ax[j].imshow(Mn, vmin=0, vmax=1, cmap='Blues'); ax[j].set_title(f"{a}, rho={CONF_RHO}: OOD-2 (row-normalized)")
    ax[j].set_xlabel('predicted'); ax[j].set_ylabel('true'); ax[j].set_xticks(range(10)); ax[j].set_yticks(range(10))
    err_rows.append(dict(arch=a, **{f"digit {d_}": f"{100*(1-Mn[d_, d_]):.1f}" for d_ in range(10)}))
plt.colorbar(im, ax=ax.ravel().tolist(), shrink=0.8); plt.savefig(f"{OUT}/fig_confusion_ood2.png", dpi=300, bbox_inches='tight'); plt.show()
T_err = pd.DataFrame(err_rows); T_err.to_csv(f"{OUT}/table_per_digit_error_ood2.csv", index=False, encoding='utf-8-sig')
print("درصد خطای هر رقم روی OOD-2:"); display(T_err)

## تفسیرپذیری: Integrated Gradients

In [ ]:
# ---------------- تفسیرپذیری: Integrated Gradients (به‌همراه کنترل Completeness) ----------------
def integrated_gradients(model, x, baseline, target, steps=64):
    model.eval(); total = torch.zeros_like(x)
    for a in torch.linspace(0, 1, steps + 1, device=x.device)[1:]:
        xi = (baseline + a * (x - baseline)).requires_grad_(True)
        g, = torch.autograd.grad(model(xi).gather(1, target[:, None]).sum(), xi)
        total += g
    return (x - baseline) * total / steps

def get_model(arch):
    if arch not in KEEP:                                           # اگر نشست قطع شده بود، دوباره آموزش بده
        KEEP[arch] = train_run(arch, DEF_DROP, build_data(0.99, SEEDS[0]), SEEDS[0])
    return KEEP[arch]

data_ig = build_data(0.99, SEEDS[0]); d = data_ig['id']
idx = torch.stack([torch.nonzero((d.y == k) & d.aligned)[0, 0] for k in range(10)])
x, y = d.rgb(idx), d.y[idx]
fig, ax = plt.subplots(5, 10, figsize=(16, 8.5))
for j in range(10): ax[0, j].imshow(x[j].permute(1, 2, 0).cpu().numpy()); ax[0, j].set_title(f"y={y[j].item()}", fontsize=9)
for r_, a in enumerate(['MLP', 'CNN3']):
    model = get_model(a); model.eval()
    ig = integrated_gradients(model, x, torch.zeros_like(x), y)                              # پایه سیاه
    with torch.no_grad(): comp = ((ig.sum((1, 2, 3))) - (model(x).gather(1, y[:, None]).squeeze(1) - model(torch.zeros_like(x)).gather(1, y[:, None]).squeeze(1))).abs().mean().item()
    xs = x[:, [1, 0, 2]]
    ig_cf = integrated_gradients(model, x, xs, y)                                            # پایه Counterfactual: همان رقم با رنگ تعویض‌شده
    with torch.no_grad(): delta = (model(x).gather(1, y[:, None]) - model(xs).gather(1, y[:, None])).squeeze(1)
    print(f"{a}: mean |completeness error| (black baseline) = {comp:.4f} | counterfactual baseline: mean Δlogit = {delta.mean().item():.3f}, "
          f"sum of IG = {ig_cf.sum((1, 2, 3)).mean().item():.3f}")
    for j in range(10):
        ax[1 + 2 * r_, j].imshow(ig[j].abs().sum(0).detach().cpu().numpy(), cmap='hot')
        ax[2 + 2 * r_, j].imshow(ig_cf[j].abs().sum(0).detach().cpu().numpy(), cmap='hot')
    ax[1 + 2 * r_, 0].set_ylabel(f"{a}: IG (black)", fontsize=8); ax[2 + 2 * r_, 0].set_ylabel(f"{a}: IG (swap)", fontsize=8)
for a_ in ax.ravel(): a_.set_xticks([]); a_.set_yticks([])
plt.suptitle("Integrated Gradients (rho=0.99 models). 'swap' baseline = same digit with swapped colors"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_integrated_gradients.png", dpi=300); plt.show()
del data_ig

## بسته‌بندی

In [ ]:
import shutil
shutil.make_archive('paper_results', 'zip', OUT)
try:
    from google.colab import files
    files.download('paper_results.zip')
except Exception:
    print('zip saved: paper_results.zip')